# 분자 특징 엔지니어링 & 전이학습 — 표현법이 성능을 만든다

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/20260825/notebooks/02_features_transfer_learning.ipynb)

**AI 신약개발 실습 · 분자 표현 & 전이학습**

같은 문제(**ESOL 수용해도 logS 회귀**)를 놓고, 분자를 **어떻게 표현하느냐**만 바꿔서 성능을 공정하게 비교합니다. 표현이 곧 모델의 상한선입니다.

1. **특징 엔지니어링(Feature Engineering)** — ECFP 지문에서 쓸모없는 비트를 걷어냅니다
   - **저분산 필터**(`VarianceThreshold`): 거의 항상 0/1인 상수성 비트 제거
   - **공선성 필터**(collinearity): |r|>0.9로 겹치는 비트 중 하나만 남김 → 차원·과적합·속도 개선
2. **전이학습(Transfer Learning)** — 대규모 SMILES로 **사전학습된 ChemBERTa**에서 임베딩을 뽑아
   그 벡터를 **전통 ML(Ridge/RandomForest)** 입력으로 사용 (feature-extraction 방식 전이학습)
3. **표현 4종 정면 비교** — 기술자 vs ECFP(raw) vs ECFP(filtered) vs **ChemBERTa 임베딩**
   (동일 train/test 분할, 동일 지표 R²/RMSE)

> ⚠️ **무-날조**: ESOL은 실제 측정 데이터, 모든 R²/RMSE·차원 수·속도는 이 노트북이 **실제로 계산**한 값입니다. RMSE는 `mean_squared_error(...) ** 0.5` 로 계산합니다.
>
> 🖥️ **실행 환경**: GPU가 있으면 자동 사용(CPU도 동작). ChemBERTa-77M은 작은 모델이라 CPU에서도 수 분 내 임베딩이 끝납니다. 임베딩은 **64개씩 배치 처리**로 메모리/속도를 관리합니다.

## 0. 설치 & 환경 설정

In [ ]:
!pip install -q rdkit scikit-learn transformers

# 한글 폰트(그래프 라벨 깨짐 방지)
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # 토크나이저 병렬 경고 억제
import matplotlib as mpl, matplotlib.font_manager as fm
_kf = [f for f in fm.findSystemFonts() if "Nanum" in f]
for _f in _kf:
    fm.fontManager.addfont(_f)
if _kf:
    mpl.rcParams["font.family"] = "NanumGothic"
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["figure.dpi"] = 120            # 선명한 그래프(dpi>=120)

# colorblind-safe 팔레트(Okabe-Ito) — 표현 4종에 일관 적용
CB = {"기술자": "#0072B2", "ECFP raw": "#E69F00",
      "ECFP filtered": "#009E73", "ChemBERTa": "#D55E00"}

import platform, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("python", platform.python_version(), "| torch", torch.__version__,
      "| device:", DEVICE, "| 한글폰트:", "OK" if _kf else "기본")


## 1. 데이터 — ESOL 수용해도 (동일 target)

Delaney(2004)의 **측정 수용해도**(logS, log mol/L). 각 분자는 SMILES로 주어집니다.
RDKit로 파싱되는 **유효 분자만** 사용합니다. 이후 모든 표현은 **이 동일한 분자 집합·동일한 target(logS)** 위에서 비교됩니다.

In [ ]:
import pandas as pd, numpy as np
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog("rdApp.*")

URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv"
df = pd.read_csv(URL)
df = df.rename(columns={"measured log solubility in mols per litre": "logS"})[["smiles", "logS"]]
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
df = df[df["mol"].notnull()].reset_index(drop=True)   # RDKit 유효 분자만
y = df["logS"].values
print("유효 분자 수:", len(df), "| logS 범위: %.2f ~ %.2f" % (y.min(), y.max()))
df[["smiles", "logS"]].head()


## 2. 기준 표현 준비 — 기술자(10D) & ECFP(raw, 2048)

- **분자 기술자**: 물성·위상 기반 해석 가능한 소수 차원(MW·logP·TPSA·HBD/HBA 등)
- **Morgan/ECFP 지문**: 반경 2·2048비트 원형 부분구조 지문(ECFP4 상당, Rogers & Hahn 2010)

ECFP는 고차원(2048)이지만 대부분의 비트가 **거의 항상 0**입니다 → §3에서 특징 엔지니어링 대상.

In [ ]:
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator

def descriptors(m):
    return [
        Descriptors.MolWt(m), Crippen.MolLogP(m), rdMolDescriptors.CalcTPSA(m),
        Lipinski.NumHDonors(m), Lipinski.NumHAcceptors(m),
        rdMolDescriptors.CalcNumRotatableBonds(m), rdMolDescriptors.CalcNumAromaticRings(m),
        rdMolDescriptors.CalcNumRings(m), Descriptors.FractionCSP3(m), m.GetNumHeavyAtoms(),
    ]
DESC_NAMES = ["MW", "logP", "TPSA", "HBD", "HBA", "RotB", "AromRings", "Rings", "FracCSP3", "HeavyAtoms"]

X_desc = np.array([descriptors(m) for m in df["mol"]], dtype=float)
mfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)   # ECFP4 상당
X_fp = np.array([list(mfp.GetFingerprint(m)) for m in df["mol"]], dtype=float)
print("기술자 행렬:", X_desc.shape, "| ECFP raw 행렬:", X_fp.shape)
print("ECFP: 켜진(=1) 비트 평균 비율 %.3f%% → 대부분 비트가 거의 상수(0)" %
      (100 * X_fp.mean()))


## 3. 특징 엔지니어링 — ECFP 지문 다이어트

훈련셋에서만 필터를 학습(누수 방지)한 뒤 train/test에 동일 적용합니다.

1. **저분산 필터** `VarianceThreshold(threshold=0.01)` — Bernoulli 비트의 분산은 `p(1-p)`.
   임계 0.01은 대략 `p<1%` 또는 `p>99%`인 **거의 상수 비트**를 제거합니다.
2. **공선성 필터** — 남은 비트들의 상관행렬 **상삼각**에서 `|r|>0.9`인 쌍은 정보가 중복되므로
   그 중 한쪽 열을 드롭합니다.

먼저 모든 표현이 공유할 **단일 train/test 분할**을 인덱스로 고정합니다(공정 비교의 핵심).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold

# 모든 표현이 공유하는 단일 분할(인덱스 기준) — 공정 비교
idx = np.arange(len(df))
idx_tr, idx_te = train_test_split(idx, test_size=0.2, random_state=42)
ytr, yte = y[idx_tr], y[idx_te]
print("train:", len(idx_tr), "| test:", len(idx_te))

Xtr_fp, Xte_fp = X_fp[idx_tr], X_fp[idx_te]

# (1) 저분산 필터 — train에서만 학습
vt = VarianceThreshold(threshold=0.01).fit(Xtr_fp)
Xtr_v, Xte_v = vt.transform(Xtr_fp), vt.transform(Xte_fp)
d_raw, d_var = X_fp.shape[1], Xtr_v.shape[1]
print(f"[저분산] {d_raw} → {d_var} 비트 ({d_raw - d_var}개 제거, 거의 상수 비트)")

# (2) 공선성 필터 — train 상관행렬 상삼각 |r|>0.9 열 드롭
corr = np.abs(np.corrcoef(Xtr_v, rowvar=False))
corr = np.nan_to_num(corr)                       # 혹시 남는 상수열 대비
upper = np.triu(corr, k=1)                        # 상삼각(대각 제외)
drop_mask = (upper > 0.9).any(axis=0)             # 앞선 어떤 열과 |r|>0.9면 드롭
keep = ~drop_mask
Xtr_filt, Xte_filt = Xtr_v[:, keep], Xte_v[:, keep]
d_filt = Xtr_filt.shape[1]
print(f"[공선성] {d_var} → {d_filt} 비트 ({int(drop_mask.sum())}개 제거, |r|>0.9 중복)")
print(f"=> 최종 차원: ECFP raw {d_raw} → filtered {d_filt} "
      f"({100 * (1 - d_filt / d_raw):.1f}% 축소)")


## 4. 원본 ECFP vs 필터링 ECFP — 과적합·속도·차원

같은 RandomForest로 **원본(2048)** 과 **필터링** ECFP를 비교합니다. 관점 3가지:
- **차원**: 특징 수 감소
- **과적합**: train R² − test R² 격차
- **속도**: 학습(fit) 시간

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

def rmse(a, b):
    return mean_squared_error(a, b) ** 0.5     # 무-날조: RMSE는 MSE의 제곱근

def bench_rf(Xtr, Xte):
    m = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    t0 = time.perf_counter(); m.fit(Xtr, ytr); fit_s = time.perf_counter() - t0
    tr_r2 = r2_score(ytr, m.predict(Xtr))
    p = m.predict(Xte)
    return dict(dim=Xtr.shape[1], train_R2=tr_r2, test_R2=r2_score(yte, p),
                RMSE=rmse(yte, p), fit_s=fit_s)

cmp_fp = pd.DataFrame([
    {"ECFP": "raw",      **bench_rf(Xtr_fp,   Xte_fp)},
    {"ECFP": "filtered", **bench_rf(Xtr_filt, Xte_filt)},
]).round({"train_R2": 3, "test_R2": 3, "RMSE": 3, "fit_s": 2})
cmp_fp["과적합(train-test)"] = (cmp_fp["train_R2"] - cmp_fp["test_R2"]).round(3)
print("=== ECFP raw vs filtered (RandomForest, 실측) ===")
cmp_fp[["ECFP", "dim", "train_R2", "test_R2", "과적합(train-test)", "RMSE", "fit_s"]]


## 5. 전이학습 — 사전학습 ChemBERTa로 SMILES 임베딩 추출

**ChemBERTa**(Chithrananda et al. 2020)는 수천만 개 SMILES로 **자기지도(MLM) 사전학습**된 RoBERTa입니다.
여기서는 `DeepChem/ChemBERTa-77M-MLM`을 **동결(frozen)** 해 SMILES → 벡터 임베딩만 뽑습니다.

- `AutoTokenizer` + `AutoModel` 로드, `last_hidden_state`를 **attention_mask 반영 mean-pool** → 분자당 1벡터
- **64개씩 배치 처리**로 OOM/속도 관리, GPU 있으면 자동 사용
- 이 벡터를 §6에서 **전통 ML(Ridge/RF)** 입력으로 사용 = **feature-extraction 전이학습**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "DeepChem/ChemBERTa-77M-MLM"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print("로드 완료:", MODEL_NAME, "| hidden size:", bert.config.hidden_size, "| device:", DEVICE)

@torch.no_grad()
def embed_smiles(smiles, batch_size=64, max_len=128):
    """SMILES 리스트 → (N, H) mean-pooled 임베딩. 배치 처리로 메모리 관리."""
    vecs = []
    for i in range(0, len(smiles), batch_size):
        chunk = smiles[i:i + batch_size]
        enc = tokenizer(chunk, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt").to(DEVICE)
        out = bert(**enc).last_hidden_state               # (B, L, H)
        mask = enc["attention_mask"].unsqueeze(-1).float()  # (B, L, 1)
        summed = (out * mask).sum(dim=1)                   # 실제 토큰만 합산
        counts = mask.sum(dim=1).clamp(min=1e-9)           # 패딩 제외 토큰 수
        vecs.append((summed / counts).cpu().numpy())       # mean-pool
        if i % (batch_size * 5) == 0:
            print(f"  임베딩 진행 {min(i + batch_size, len(smiles))}/{len(smiles)}")
    return np.vstack(vecs)

X_bert = embed_smiles(df["smiles"].tolist(), batch_size=64)
Xtr_bert, Xte_bert = X_bert[idx_tr], X_bert[idx_te]
print("ChemBERTa 임베딩 행렬:", X_bert.shape)


## 6. 표현 4종 정면 비교 — 동일 train/test, 동일 지표

**기술자(10D) · ECFP raw(2048) · ECFP filtered · ChemBERTa 임베딩**을
각각 **Ridge**(선형)와 **RandomForest**(비선형)에 넣어 logS를 예측합니다.
- Ridge는 스케일에 민감 → `StandardScaler`를 파이프라인에 포함
- ECFP는 희소 이진, ChemBERTa/기술자는 밀집 실수 — 표현별 궁합 차이를 관찰

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

REPS = {
    "기술자":        (X_desc[idx_tr],  X_desc[idx_te]),
    "ECFP raw":      (Xtr_fp,          Xte_fp),
    "ECFP filtered": (Xtr_filt,        Xte_filt),
    "ChemBERTa":     (Xtr_bert,        Xte_bert),
}
DIMS = {k: v[0].shape[1] for k, v in REPS.items()}

def run(Xtr, Xte, model):
    model.fit(Xtr, ytr); p = model.predict(Xte)
    return r2_score(yte, p), rmse(yte, p), p

rows, preds = [], {}
for name, (Xtr, Xte) in REPS.items():
    r2_ridge, rmse_ridge, p_ridge = run(
        Xtr, Xte, make_pipeline(StandardScaler(), Ridge(alpha=1.0)))
    r2_rf, rmse_rf, p_rf = run(
        Xtr, Xte, RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
    rows.append({"표현": name, "차원": DIMS[name],
                 "Ridge R²": round(r2_ridge, 3), "Ridge RMSE": round(rmse_ridge, 3),
                 "RF R²": round(r2_rf, 3), "RF RMSE": round(rmse_rf, 3)})
    preds[(name, "Ridge")] = (r2_ridge, rmse_ridge, p_ridge)
    preds[(name, "RF")] = (r2_rf, rmse_rf, p_rf)

board = pd.DataFrame(rows)
print("=== 표현 × 모델 성능 (동일 test셋, 실측) ===")
board


In [ ]:
# 최고 성능 조합(전체에서 test R² 최대) 자동 선정
best_key = max(preds, key=lambda k: preds[k][0])
best_r2, best_rmse, best_pred = preds[best_key]
print(f"최고 조합: {best_key[0]} + {best_key[1]}  →  R²={best_r2:.3f}, RMSE={best_rmse:.3f}")

# 표현별 최고(두 모델 중 더 좋은 R²)도 요약
best_per_rep = {name: max(preds[(name, "Ridge")][0], preds[(name, "RF")][0]) for name in REPS}
for k, v in best_per_rep.items():
    print(f"  - {k:14s} best R² = {v:.3f}")


## 7. 시각화 — 차원 축소 · 표현별 성능 · 예측-실측

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# (1) ECFP 특징 엔지니어링: 차원 축소 막대
stages = ["raw\n(2048)", "저분산\n필터 후", "공선성\n필터 후"]
dvals = [d_raw, d_var, d_filt]
bars = axes[0].bar(stages, dvals, color=["#E69F00", "#56B4E9", "#009E73"])
axes[0].set_ylabel("특징(비트) 수"); axes[0].set_title("ECFP 특징 엔지니어링: 차원 축소")
for b, v in zip(bars, dvals):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 20, str(v), ha="center", fontsize=10)

# (2) 표현별 성능(두 모델 중 더 좋은 test R²) 막대
names = list(REPS.keys())
vals = [best_per_rep[n] for n in names]
colors = [CB[n] for n in names]
bars = axes[1].bar(names, vals, color=colors)
axes[1].set_ylabel("test R² (표현별 best)"); axes[1].set_title("표현별 성능 비교 (logS 회귀)")
axes[1].set_ylim(0, 1); axes[1].tick_params(axis="x", rotation=15)
for b, v in zip(bars, vals):
    axes[1].text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

# (3) 예측 vs 실측 (전체 최고 조합)
axes[2].scatter(yte, best_pred, s=16, alpha=0.5, edgecolor="k", linewidth=0.3, c="#0072B2")
lim = [min(yte.min(), best_pred.min()), max(yte.max(), best_pred.max())]
axes[2].plot(lim, lim, "r--", lw=1)
axes[2].set_xlabel("실측 logS"); axes[2].set_ylabel("예측 logS")
axes[2].set_title(f"예측 vs 실측 (best: {best_key[0]}+{best_key[1]})\n"
                  f"R²={best_r2:.3f}  RMSE={best_rmse:.3f}")
plt.tight_layout(); plt.show()


### (선택) ChemBERTa 임베딩 구조 시각화 — PCA (UMAP 있으면 함께)

사전학습 임베딩이 logS와 연속적으로 정렬되어 있는지 2D로 투영해 봅니다.
색상이 logS로 매끄럽게 변한다면, 그 표현이 이미 용해도 관련 정보를 담고 있다는 신호입니다.

In [ ]:
from sklearn.decomposition import PCA

emb2_pca = PCA(n_components=2, random_state=42).fit_transform(X_bert)

# UMAP은 있으면 사용, 없으면 PCA만
proj = [("PCA", emb2_pca)]
try:
    import umap  # noqa: F401
    from umap import UMAP
    emb2_umap = UMAP(n_components=2, random_state=42).fit_transform(X_bert)
    proj.append(("UMAP", emb2_umap))
except Exception as e:
    print("UMAP 미설치/미사용 → PCA만 표시 (%s)" % type(e).__name__)

fig, axes = plt.subplots(1, len(proj), figsize=(6.2 * len(proj), 5), squeeze=False)
for ax, (title, emb2) in zip(axes[0], proj):
    sc = ax.scatter(emb2[:, 0], emb2[:, 1], c=y, cmap="viridis", s=14, alpha=0.7)
    ax.set_title(f"ChemBERTa 임베딩 {title} (색=logS)")
    ax.set_xlabel(f"{title}-1"); ax.set_ylabel(f"{title}-2")
    plt.colorbar(sc, ax=ax, fraction=0.046, label="logS")
plt.tight_layout(); plt.show()


## 8. 개념 정리 — 자기지도 사전학습 표현을 전통 ML로 '전이'

이 노트북의 §5–6이 바로 **자기지도 사전학습 표현으로 특징을 뽑아 전통 ML에 전이**하는 방식입니다.
라벨이 없는 방대한 SMILES로 먼저 표현(임베딩)을 학습해 두면, 라벨이 적은 다운스트림 과제(여기서는 logS 회귀)에서
그 임베딩을 **고정 특징(feature extractor)** 으로 재사용해 Ridge/RandomForest 같은 가벼운 모델만 학습하면 됩니다.
전체 딥러닝 모델을 fine-tuning하지 않고도, 사전학습이 담아 둔 화학 지식을 그대로 빌려 쓰는 셈입니다.

**MolCLR (그래프 기반 동종 접근).** MolCLR(Wang et al., *Nature Machine Intelligence* 2022; 저장소 `yuyangw/MolCLR`)은
SMILES 시퀀스가 아니라 **분자 그래프**를 대상으로 그래프신경망(GNN)을 **대조학습(contrastive learning)** 으로 자기지도 사전학습합니다.
원자 마스킹·결합 삭제·부분구조 제거 같은 그래프 증강으로 같은 분자의 두 뷰를 가깝게, 다른 분자를 멀게 학습해 범용 분자 표현을 얻고,
이를 다운스트림 물성/활성 예측에 전이합니다 — 본 실습의 ChemBERTa(시퀀스 기반)와 **표현 modality만 다른 같은 철학**의 전이학습입니다.
다만 MolCLR은 그래프 GNN 스택(예: PyTorch Geometric) 설치가 무거워, 이 실습에서는 설치가 가벼운 **SMILES 기반 ChemBERTa로 시연**하고
MolCLR은 개념(선택·고급)으로만 소개합니다.

## 9. 정리

- **표현이 상한을 만든다**: 동일 target(logS)·동일 분할에서도 기술자 vs ECFP vs ChemBERTa의 R²/RMSE가 서로 다릅니다(위 표 실측).
- **특징 엔지니어링은 공짜 점심에 가깝다**: 저분산+공선성 필터로 ECFP 차원을 크게 줄이면 **속도·과적합**이 개선되며 성능은 대체로 유지됩니다(§4 실측 격차 비교).
- **전이학습**: 사전학습 ChemBERTa 임베딩을 전통 ML에 넣는 feature-extraction 방식만으로도 경쟁력 있는 표현을 손쉽게 확보할 수 있습니다.
- **다음 단계**: 임베딩 fine-tuning, scaffold split 검증, MolCLR 등 그래프 사전학습 표현과의 비교.

> ⚠️ 교육용 데모입니다. 실제 적용은 scaffold split·외부검증·불확실성 정량이 필요하며, 예측은 실험 검증 전까지 결론이 아닙니다. 모든 수치는 이 노트북이 실제로 계산한 값입니다.

**참고문헌 (실재)**
- Chithrananda, Grand & Ramsundar. *ChemBERTa: Large-Scale Self-Supervised Pretraining for Molecular Property Prediction.* arXiv:2010.09885 (2020).
- Wang, Wang, Cao, Farimani. *Molecular Contrastive Learning of Representations via Graph Neural Networks (MolCLR).* Nature Machine Intelligence 4:279–287 (2022). 저장소: github.com/yuyangw/MolCLR
- Rogers & Hahn. *Extended-Connectivity Fingerprints.* J Chem Inf Model 50:742–754 (2010).
- Delaney. *ESOL: Estimating Aqueous Solubility Directly from Molecular Structure.* J Chem Inf Comput Sci 44:1000–1005 (2004).
- RDKit: Open-source cheminformatics (https://www.rdkit.org)